# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuan05-afk/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

Author: Yuan Andrei C. Mariano · Lane: **Refresh / Content Opportunity Scoring**

> Skills used: `framing-ml-problems` + `flyrank/flyrank-data`.


## 1. My lane (or freestyle) and why

**Lane:** Refresh / Content Opportunity Scoring (core lane).

I am ranking existing content items for editor review: which pages should be refreshed first,
given observable search and engagement signals. This matches the starter pipeline's unit
(one row = one content item), output shape (ranked queue with scores, actions, reason codes),
and decision (an editor prioritizes rewrite effort). Freestyle is not needed; the core lane
already maps to a measurable ranking/scoring task with a clear cost of a wrong call.


In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

ROOT = Path('.').resolve()
# Support running from repo root or work/notebooks
if (ROOT / 'data/raw/content_refresh_anonymized.csv').exists():
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
elif (ROOT.parents[1] / 'data/raw/content_refresh_anonymized.csv').exists():
    ROOT = ROOT.parents[1]
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
else:
    # Colab / nested
    for cand in [ROOT, *ROOT.parents]:
        if (cand / 'data/raw/content_refresh_anonymized.csv').exists():
            ROOT = cand
            DATA = cand / 'data/raw/content_refresh_anonymized.csv'
            break
    else:
        raise FileNotFoundError('data/raw/content_refresh_anonymized.csv')

WORK_OUT = ROOT / 'work' / 'outputs'
WORK_OUT.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA)
print(f'Loaded {len(df):,} rows × {df.shape[1]} cols from {DATA.relative_to(ROOT)}')
print(f'Clients (pseudonymous): {df["client_id"].nunique()}')

# Supporting context for the lane choice
print('trend_direction mix:')
print((df['trend_direction'].value_counts(normalize=True)*100).round(1).astype(str) + '%')


Loaded 30,000 rows × 44 cols from data\raw\content_refresh_anonymized.csv
Clients (pseudonymous): 32
trend_direction mix:
trend_direction
down      54.2%
stable    19.9%
up        14.6%
new        7.5%
flat       3.8%
Name: proportion, dtype: str


## 2. The question: decision, action, cost of a wrong call

**Decision:** Which pages should an editor open for refresh review this week?

**Action:** Prioritize a short review queue (top-K), then rewrite or expand only those pages.

**Cost of a wrong call:**
- False positive (queue a stable page): wasted rewrite hours and opportunity cost.
- False negative (miss a declining visible page): continued impression/click loss while
  editorial time goes elsewhere.

**Why data/ML helps:** Decline, staleness, CTR gaps, and visibility interact. A single
threshold rule catches some cases but mixes thin pages, CTR problems, and stale high-impression
pages. A scored ranked queue is decision-support, not an automatic publish trigger.


In [2]:
# Cost framing as base rates (decision-support context, not a causal claim)
n = len(df)
down_rate = (df['trend_direction'] == 'down').mean()
print(f'Pages: {n:,}')
print(f'Observed declining (trend_direction=down): {down_rate:.1%} — base rate any top-K must beat')
print(f'Median days_since_last_update: {df["days_since_last_update"].median():.0f}')
print(f'Median impressions_90d: {df["impressions_90d"].median():.0f}')


Pages: 30,000
Observed declining (trend_direction=down): 54.2% — base rate any top-K must beat
Median days_since_last_update: 20
Median impressions_90d: 731


## 3. Quick look at the data (2-3 real numbers)

Three measured facts that justify staying on this lane for the rest of the track.


In [3]:
facts = {
    'rows': int(len(df)),
    'clients': int(df['client_id'].nunique()),
    'pct_down': round(float((df['trend_direction']=='down').mean())*100, 1),
    'pct_stale_180_and_imp500': round(float(
        ((df['days_since_last_update']>=180) & (df['impressions_90d']>=500)).mean()
    )*100, 1),
    'median_avg_position_known': round(float(df.loc[df['avg_position']>0, 'avg_position'].median()), 1),
}
print(json.dumps(facts, indent=2))
print()
print(f"1) {facts['pct_down']}% of pages show trend_direction=down — decline is common enough that")
print('   a random queue is a weak editor default.')
print(f"2) {facts['pct_stale_180_and_imp500']}% are stale (≥180d) AND still visible (≥500 imp/90d) —")
print('   refresh candidates with real demand.')
print(f"3) Median known avg_position is {facts['median_avg_position_known']} — many pages sit")
print('   outside easy page-one CTR assumptions, so ranking needs more than one flag.')
(WORK_OUT / 'w01_lane_facts.json').write_text(json.dumps(facts, indent=2))


{
  "rows": 30000,
  "clients": 32,
  "pct_down": 54.2,
  "pct_stale_180_and_imp500": 0.1,
  "median_avg_position_known": 11.4
}

1) 54.2% of pages show trend_direction=down — decline is common enough that
   a random queue is a weak editor default.
2) 0.1% are stale (≥180d) AND still visible (≥500 imp/90d) —
   refresh candidates with real demand.
3) Median known avg_position is 11.4 — many pages sit
   outside easy page-one CTR assumptions, so ranking needs more than one flag.


128

## 4. Careful words: what I can and can't claim

**Can claim (observed / measured / directional / decision-support):**
- Measured associations between features and an observed decline label on a client-holdout split.
- Directional ranking quality (e.g. Precision@K lift over a transparent baseline).
- A review queue suitable as editor decision-support.

**Cannot claim:**
- That the model predicts Google's algorithm or causes ranking recovery.
- Causal proof that refreshing a page will reverse decline (no experiment design here).
- Performance on private client URLs or warehouse rows I have not queried yet.

Warehouse note: Hugging Face `FlyRank/internship-warehouse` access is deferred if no token is
available; this track uses the starter CSV honestly and documents SETUP steps for later warehouse work.


In [4]:
print('Claim ladder for this lane:')
print('- observed: trend_direction / impressions / freshness on starter CSV')
print('- measured: Precision@K on client-holdout vs baseline')
print('- directional: higher score → higher chance of decline label in holdout')
print('- decision-support: ranked refresh queue for human review')
print('- not claimed: causal recovery, Google ranking prediction')


Claim ladder for this lane:
- observed: trend_direction / impressions / freshness on starter CSV
- measured: Precision@K on client-holdout vs baseline
- directional: higher score → higher chance of decline label in holdout
- decision-support: ranked refresh queue for human review
- not claimed: causal recovery, Google ranking prediction


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
